In [ ]:
import requests
import json
import time
from pathlib import Path

# Endpoint ArcGIS BIG
url = (
    "https://kspservices.big.go.id/satupeta/rest/services/"
    "RBI/Administrasi_AR_KabKota_50K/MapServer/0/query"
)

session = requests.Session()

# =====================================
# 1. Ambil seluruh Object ID
# =====================================

params = {
    "where": "1=1",
    "returnIdsOnly": "true",
    "f": "json"
}

response = session.get(url, params=params, timeout=120)
response.raise_for_status()

id_data = response.json()

if "error" in id_data:
    raise Exception(id_data["error"])

object_ids = id_data.get("objectIds", [])

print("Total Object ID:", len(object_ids))

if not object_ids:
    raise Exception("Object ID tidak ditemukan.")

# =====================================
# 2. Download polygon secara bertahap
# =====================================

all_features = []
batch_size = 20

for start in range(0, len(object_ids), batch_size):

    batch_ids = object_ids[start:start + batch_size]

    params = {
        "objectIds": ",".join(map(str, batch_ids)),
        "outFields": "*",
        "returnGeometry": "true",
        "outSR": "4326",
        "f": "geojson"
    }

    response = session.get(url, params=params, timeout=180)
    response.raise_for_status()

    result = response.json()

    if "error" in result:
        raise Exception(result["error"])

    features = result.get("features", [])

    all_features.extend(features)

    print(
        f"Progress: {min(start + batch_size, len(object_ids))}"
        f"/{len(object_ids)} | "
        f"Berhasil: {len(features)} fitur"
    )

    time.sleep(0.5)

# =====================================
# 3. Simpan menjadi GeoJSON
# =====================================

geojson = {
    "type": "FeatureCollection",
    "features": all_features
}

output_dir = Path("data/raw")
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "batas_kabupaten_kota.geojson"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(geojson, f, ensure_ascii=False)

print("\nDOWNLOAD SELESAI")
print("Total Object ID:", len(object_ids))
print("Total fitur:", len(all_features))
print("File:", output_file)
print("Ukuran MB:", round(output_file.stat().st_size / 1024**2, 2))

In [ ]:
import geopandas as gpd

gdf = gpd.read_file(
    "data/raw/batas_kabupaten_kota.geojson"
)

print("Jumlah fitur:", len(gdf))
print("Jenis geometri:")
print(gdf.geometry.geom_type.value_counts())

print("\nDaftar kolom:")
print(gdf.columns.tolist())

print("\nContoh data:")
display(gdf.head())
